# MODIS + VIIRS hotspot harmonization (Algeria, pilot)

Goal: one consistent monthly hotspot series from 2000 to today, so the sensor of origin no longer matters.

Pipeline: load and crop -> inspect -> raw comparison (the "before") -> clean and aggregate -> calibrate in the overlap -> harmonized series -> export for the app.

Status: skeleton. Column names follow the FIRMS archive format; check them against your real files in section 1.

## 0. Setup and parameters

All choices that affect results live here, so you can change them in one place and rerun. Each one should be justified in your write-up.

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path("../data/raw")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

# Pilot area: northern Algeria (wildfires), avoids most Saharan gas flares
LAT_MIN, LAT_MAX = 34.0, 37.2
LON_MIN, LON_MAX = -2.3, 8.7

# Confidence rules (check the value formats in section 1)
MODIS_MIN_CONF = 30           # MODIS confidence is numeric 0-100
VIIRS_KEEP_CONF = {"n", "h"}  # VIIRS confidence is l / n / h

# Spatial units
DEDUP_CELL = 0.01   # ~1 km, used to count "occupied cells per day"
GRID = 0.25         # grid of the final monthly product, in degrees

OVERLAP_START = "2013-01-01"  # first full month with VIIRS
LAST_FULL_MONTH = "2026-10-01"  # exclude the current, incomplete month

## 1. Load and crop

Question answered: what is in the files, and can I fit them in memory?
Load only the columns we need and crop to the pilot area immediately, since everything after this is smaller and faster.

In [8]:
USECOLS = ["latitude", "longitude", "acq_date", "acq_time", "satellite",
           "confidence", "frp", "daynight", "scan", "track"]

def load_sensor(pattern):
    # FIRMS delivers two files per sensor: *archive* (standard processing, historical)
    # and *nrt* (near real time, the most recent months). We load both.
    parts = []
    for f in sorted(RAW.glob(pattern)):          # "archive" sorts before "nrt"
        df = pd.read_csv(f, usecols=lambda c: c in USECOLS + ["type"],
                         dtype={"satellite": "category", "daynight": "category"})
        df = df[df.latitude.between(LAT_MIN, LAT_MAX) & df.longitude.between(LON_MIN, LON_MAX)]
        df["acq_date"] = pd.to_datetime(df["acq_date"])
        print(f"{f.name:32s} {len(df):>9,d} rows | {df.acq_date.min().date()} -> {df.acq_date.max().date()}")
        parts.append(df)
    out = pd.concat(parts, ignore_index=True)
    # If the two files overlap in time, the same detection would appear twice.
    # Keep the first copy, which is the archive (standard processing) version.
    before = len(out)
    out = out.drop_duplicates(subset=["latitude", "longitude", "acq_date", "acq_time", "satellite"], keep="first")
    print(f"  duplicates removed across files: {before - len(out):,d}")
    return out

modis = load_sensor("modis*.csv")
viirs = load_sensor("viirs*.csv")
modis.to_parquet(OUT / "modis_north.parquet", index=False)
viirs.to_parquet(OUT / "viirs_north.parquet", index=False)


modis_archive.csv                   93,215 rows | 2000-11-02 -> 2026-08-31
modis_nrt.csv                          284 rows | 2026-09-01 -> 2026-11-06
  duplicates removed across files: 0
viirs_archive.csv                  304,565 rows | 2012-07-04 -> 2026-07-31
viirs_nrt.csv                        9,540 rows | 2026-08-01 -> 2026-11-02
  duplicates removed across files: 0


In [ ]:
# Reload from parquet on later runs (much faster than CSV)
# modis = pd.read_parquet(OUT / "modis_north.parquet")
# viirs = pd.read_parquet(OUT / "viirs_north.parquet")

for name, d in [("MODIS", modis), ("VIIRS", viirs)]:
    print(name, "| rows:", f"{len(d):,d}", "| dates:", d.acq_date.min().date(), "->", d.acq_date.max().date())
    print("  confidence values:", d["confidence"].astype(str).unique()[:8])
    print("  satellites:", d["satellite"].unique().tolist())
    if "type" in d.columns:
        print("  type values:", d["type"].unique().tolist(), "(check FIRMS docs for meaning)")

## 2. Raw comparison (the "before" picture)

Question answered: how different do the two sensors look before any correction?
This becomes the first figure of your demo and your write-up.

In [ ]:
def monthly_raw(d):
    return d.groupby(d.acq_date.dt.to_period("M")).size()

raw_m, raw_v = monthly_raw(modis), monthly_raw(viirs)
ax = raw_m.plot(label="MODIS (raw detections)", figsize=(11, 4))
raw_v.plot(ax=ax, label="VIIRS (raw detections)")
ax.set_ylabel("detections per month"); ax.legend(); ax.set_title("Raw monthly detections")
plt.show()

## 3. Clean and aggregate

Three steps, each with a reason:
1. **Confidence filter**, so low-quality detections do not inflate counts.
2. **Occupied-cell counting**: VIIRS pixels are smaller, so raw counts are not comparable. Counting unique ~1 km cells with a detection per day reduces the resolution gap.
3. **Deduplication across satellites**: the same fire seen by two satellites on the same day counts once.

In [ ]:
def clean(d, sensor):
    d = d.copy()
    conf = d["confidence"].astype(str).str.lower()
    if sensor == "modis":
        d = d[pd.to_numeric(conf, errors="coerce") >= MODIS_MIN_CONF]
    else:
        d = d[conf.isin(VIIRS_KEEP_CONF)]
    # Optional: keep only vegetation fires if a 'type' column exists (check docs for the code)
    # if "type" in d.columns: d = d[d["type"] == 0]
    return d

def occupied_cells(d):
    d = d.copy()
    d["clat"] = (d.latitude / DEDUP_CELL).round().astype(int)
    d["clon"] = (d.longitude / DEDUP_CELL).round().astype(int)
    # one row per (cell, day), regardless of how many satellites saw it
    return d.drop_duplicates(subset=["clat", "clon", "acq_date"])

def to_monthly_grid(d):
    d = d.copy()
    d["lat"] = (np.floor(d.latitude / GRID) * GRID + GRID / 2).round(3)
    d["lon"] = (np.floor(d.longitude / GRID) * GRID + GRID / 2).round(3)
    d["year"], d["month"] = d.acq_date.dt.year, d.acq_date.dt.month
    return d.groupby(["lat", "lon", "year", "month"], as_index=False).size().rename(columns={"size": "count"})

m_clean = occupied_cells(clean(modis, "modis"))
v_clean = occupied_cells(clean(viirs, "viirs"))
print("MODIS:", len(modis), "->", len(m_clean), "| VIIRS:", len(viirs), "->", len(v_clean))

grid_m, grid_v = to_monthly_grid(m_clean), to_monthly_grid(v_clean)

## 4. Calibrate in the overlap

Question answered: how do monthly VIIRS counts relate to monthly MODIS counts when both see the same fires?
Start with the simplest model, a single ratio. Validate by leaving out whole years, because neighbouring months and fires are correlated and a random split would flatter the result.

In [ ]:
def region_monthly(g):
    s = g.groupby(["year", "month"])["count"].sum()
    s.index = pd.to_datetime(dict(year=s.index.get_level_values(0), month=s.index.get_level_values(1), day=1))
    return s

sm, sv = region_monthly(grid_m), region_monthly(grid_v)
both = pd.concat([sm.rename("modis"), sv.rename("viirs")], axis=1).dropna()
both = both[(both.index >= OVERLAP_START) & (both.index <= LAST_FULL_MONTH)]
print(len(both), "overlap months")
both.plot.scatter(x="modis", y="viirs", figsize=(5, 5), title="Monthly counts, overlap period")

In [ ]:
# Leave-one-year-out validation of the ratio model: viirs ~ ratio * modis
years = sorted(set(both.index.year))
errors = []
for y in years:
    train, test = both[both.index.year != y], both[both.index.year == y]
    ratio = train.viirs.sum() / train.modis.sum()
    pred = ratio * test.modis
    errors.append(pd.DataFrame({"year": y, "obs": test.viirs, "pred": pred}))
cv = pd.concat(errors)
rmse = np.sqrt(((cv.obs - cv.pred) ** 2).mean())
mape = ((cv.obs - cv.pred).abs() / cv.obs.replace(0, np.nan)).median()
print(f"leave-one-year-out RMSE: {rmse:,.0f} | median relative error: {mape:.1%}")

RATIO = both.viirs.sum() / both.modis.sum()   # final ratio, fitted on all overlap months
print("final ratio VIIRS/MODIS:", round(RATIO, 3))

## 5. Harmonized series and export

Before the overlap we use MODIS scaled to the VIIRS level; from the overlap start we use VIIRS directly.
The export has the exact columns the app expects: `lat, lon, year, month, count` (plus `source` for transparency).

In [ ]:
cut = pd.Timestamp(OVERLAP_START)
def with_date(g): return g.assign(date=pd.to_datetime(dict(year=g.year, month=g.month, day=1)))

gm, gv = with_date(grid_m), with_date(grid_v)
gm = gm[(gm.date < cut)].copy();  gm["count"] = (gm["count"] * RATIO).round().astype(int); gm["source"] = "MODIS (scaled)"
gv = gv[(gv.date >= cut) & (gv.date <= LAST_FULL_MONTH)].copy();                gv["source"] = "VIIRS"

harmonized = pd.concat([gm, gv], ignore_index=True)[["lat", "lon", "year", "month", "count", "source"]]
harmonized.to_parquet(OUT / "monthly_grid.parquet", index=False)
print(len(harmonized), "rows written to", OUT / "monthly_grid.parquet")

## 6. Sanity checks (the "after" picture)

Question answered: is the harmonized series continuous, and do the peaks fall in the fire season?
Look for a visible jump at the 2013 switch; a jump means the calibration is off.

In [ ]:
series = harmonized.groupby(["year", "month"])["count"].sum()
series.index = pd.to_datetime(dict(year=series.index.get_level_values(0), month=series.index.get_level_values(1), day=1))
series.plot(figsize=(11, 4), title="Harmonized monthly hotspot count")
plt.axvline(cut, color="grey", linestyle="--"); plt.show()

# Seasonal check: which months peak?
print(harmonized.groupby("month")["count"].sum().sort_values(ascending=False).head(4))

## Next steps
- Point `app/app.py` at `data/processed/monthly_grid.parquet`.
- Try per-month or per-region ratios if the single ratio leaves a clear seasonal error.
- Add confidence intervals from the leave-one-year-out errors to the app.
- Document every parameter from section 0 in the README.